In [1]:
import pandas as pd

""" Sandobx for all things realted Trialpop - or holding of (chans, trials x times)
- i.e,, dealing with differtnet length trials, without taking snippets.
"""


' Sandobx for all things realted Trialpop - or holding of (chans, trials x times)\n- i.e,, dealing with differtnet length trials, without taking snippets.\n'

In [2]:
%load_ext autoreload
%autoreload 2

from neuralmonkey.classes.session import load_mult_session_helper
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

#%matplotlib inline

In [ ]:
# DATE = 220830
# animal = "Pancho"
# DATE = 230615
# animal = "Diego"
# spikes_version = "tdt"

# DATE = 221031
# animal = "Pancho"

DATE = 220715
animal = "Pancho"

spikes_version = "kilosort_if_exists"
MS = load_mult_session_helper(DATE, animal, MINIMAL_LOADING=True, spikes_version=spikes_version) 
# MS = load_mult_session_helper(DATE, animal, MINIMAL_LOADING=True, spikes_version="tdt") 

In [ ]:
#### TODO:

In [ ]:
# problem 1. Stroke onset time -- why don't they match between SN and Datasetbeh.
# problem 2. concatting DS across sessions for Pancho --> failing.
# NOTE: I think these have been solved?

# Things that work on all data (not single PA)

In [ ]:
# (1) Do PA-related preprocessing
from neuralmonkey.classes.population_mult import dfpa_concat_normalize_fr_split_multbregion_flex, dfpa_concat_normalize_fr_split_singlebregion, dfpa_concat_normalize_fr_split_multbregion
fr_mean_subtract_method = "across_time_bins"
# fr_mean_subtract_method = "each_time_bin"
PLOT=False

pa = DFTRIAL["pa"].values[0]
pa.plotNeurHeat(0)

print(" == (5) Sqrt transform")
for pa in DFTRIAL["pa"]:
    pa.X = pa.X**0.5

dfpa_concat_normalize_fr_split_multbregion_flex(DFTRIAL, fr_mean_subtract_method, PLOT)

pa = DFTRIAL["pa"].values[0]
pa.plotNeurHeat(0)

In [ ]:
# TODO, all the cleanup
# (1) Do PA-related preprocessing
from neuralmonkey.classes.population_mult import dfpa_concat_normalize_fr_split_multbregion_flex, dfpa_concat_normalize_fr_split_singlebregion
fr_mean_subtract_method = "across_time_bins"
# fr_mean_subtract_method = "each_time_bin"
PLOT=False

pa = DFTRIAL["pa"].values[0]
pa.plotNeurHeat(0)

dfpa_concat_normalize_fr_split_multbregion_flex(DFTRIAL, fr_mean_subtract_method, PLOT)

pa = DFTRIAL["pa"].values[0]
pa.plotNeurHeat(0)

In [ ]:
# (3) Do cleanup of trials/chans
from neuralmonkey.classes.population_mult import dfallpa_preprocess_sitesdirty_check_if_preprocessed, dfallpa_preprocess_sitesdirty_single
dfallpa_preprocess_sitesdirty_check_if_preprocessed(DFTRIAL, animal, DATE)

In [ ]:
# (2) Do dim reduction=

# TODO: Concatenate all data into frmat, then do dim reduction

pa = DFTRIAL["pa"].values[0]

scalar_or_traj = "traj"
dim_red_method = "pca"
savedir = "/tmp"
twind_pca = (pa.Times[0], pa.Times[-1])
tbin_dur = "default"
tbin_slide = "default"
# tbin_dur = 0.1
# tbin_slide = 0.05

pa.dataextract_dimred_wrapper(scalar_or_traj, dim_red_method, savedir, 
                                   twind_pca, tbin_dur, tbin_slide, 
                                   NPCS_KEEP = 10)


# Good methods

In [ ]:
len(MS.sitegetter_all(how_combine="intersect"))

In [ ]:
for pa in MS.SessionsList[0].PopAnalDict.values():
    print(pa.Chans)

In [ ]:
# This will take a few minutes
PT = MS.trialpop_extract_wrapper(remove_bloat=True)

In [ ]:
# PT.Dat holds all data, 
# - where each row is a trial.
# - SN is there mostly for its helper functions and its saved event timings.
# - pa is popuilation activity holding all channels.
# - time_start_incl_flank, timestamp in seconds, within that trial.
# - spike_times, dict, holding spikes for each trial
# - event_times, dict, holding times of events on that trial
PT.Dat

In [ ]:
# Anywhere with variable "chan" refers to these:
PT.Chans[:5]

In [ ]:
# And info for each chan:
PT.LabelChans[:5]

In [ ]:
# Trials are labeled by index (0, 1, ..) or trialcode
PT.Trials[:5]

In [ ]:
# Behavioral data for each trial
dfbeh = PT.datasetbeh_extract_df_trialaligned()
dfbeh[:5]

### Helper methods to extract data

In [ ]:
# Session object (and its associated trial_session)
idxtrial = 100
sn, trialsn = PT.extract_sn(idxtrial)

# session has many metods. some will fail since the bulky data have been consoliadted into PT. You can try

In [ ]:
# Spike times
chans = PT.info_bregion_to_chans("M1_m")
PT.extract_spike_times(idxtrial, chans[0])

chans = PT.LabelChans[PT.LabelChans["region"] == "PMv_l"]["chan_pt"].tolist()
idxtrial = 100
pa = PT.extract_pa(idxtrial)
paslice = pa.slice_by_dim_values_wrapper("chans", chans)



In [ ]:
# channels for a given bregion
PT.info_bregion_to_chans("PMv_m")

In [ ]:
# Behavioral dataset, rows match self.Trials
D = PT.datasetbeh_extract_D()
assert np.all(D.Dat["trialcode"] == PT.Dat["trialcode"])
D.plotSingleTrial(100);

In [ ]:
D.plotMultTrials([10, 20, 30, 49])

### Methods to plot rasters

In [ ]:
# Plot one trial, mult chans as rows
idx_trial = 100
chans = PT.info_bregion_to_chans("M1_l")
fig, axes, _ = PT.plot_raster_create_figure_blank(10, len(chans), 1, 1)
ax = axes.flatten()[0]
PT.plot_raster_multchans(idxtrial, chans, ax)

In [ ]:
# Plot one chan, mult trials as rows
list_idxtrials = list(range(30))
chan = PT.info_bregion_to_chans("FP_p")[6]
fig, axes, _ = PT.plot_raster_create_figure_blank(10, len(list_idxtrials), 1, 1)
ax = axes.flatten()[0]
alignto = "samp"
PT.plot_raster_multtrials(list_idxtrials, chan, ax, alignto, overlay_trial_events=True)

In [ ]:
# As an exmaple, you could plot rasters just for trials passing certain conditions
dfbeh = PT.datasetbeh_extract_df_trialaligned()
idxtrials = dfbeh[dfbeh["aborted"] == True].index.tolist()[:100] # just trials with abort
fig, axes, _ = PT.plot_raster_create_figure_blank(10, len(list_idxtrials), 2, 5)
for ax, chan in zip(axes.flatten(), PT.info_bregion_to_chans("FP_a")[:10]):
    PT.plot_raster_multtrials(idxtrials, chan, ax, alignto="post")

In [ ]:
# Plot a single raster, with conditions as diff y-axis blocks.
event = "post"
vars_group =  ["supervision_stage_concise", "aborted"]
chan = chans[0]
overlay_trial_events = True
PT.plot_rasters_block_by_var(chan, vars_group, event, overlay_trial_events)

In [ ]:
# Same as above, but only plot trials with at least this much time (data) after the event
remaintime_needed = 2
event = "post"
idxs_keep = []
for idxtrial in range(len(PT.Trials)):
    
    evtime = PT.extract_event_times(idxtrial)[event][0]
    totaltime = PT.Dat.iloc[idxtrial]["time_off_incl_flank"]
    
    remaintime = totaltime - evtime

    if remaintime>remaintime_needed:
        idxs_keep.append(idxtrial)

#TODO: prune PT and then plot again as above.

### Methods to plot smoothed firing rates

In [ ]:
# For each channel, plots its firing rate split in various ways
event = "samp"
PA = PT.pa_extract_align_to_event(event, 0.5, 0.5)
chans = PT.info_bregion_to_chans("FP_a")
for ch in chans[:5]:
    PA.plotwrapper_smoothed_fr_split_by_label_and_subplots(ch, "aborted", ["epoch"])

### Extract PA

In [ ]:
# Can do useful things using PA methods.
time_pre = 0.5
time_post = 0.8
event = "samp"
PA = PT.pa_extract_align_to_event(event, time_pre, time_post)

### Save PT

In [ ]:
savedir = f"/lemur2/lucas/Dropbox/SCIENCE/FREIWALD_LAB/DATA/PopanalTrial"
PT.save(savedir, suffix=f"{animal}-{DATE}")

### Testing out file sizes
# # sn = PT.Dat["SN"].values[0]
# sn = PT.Dat["pa"].tolist()

# import pickle
# with open("/tmp/sn.pkl", "wb") as f:
#     pickle.dump(sn, f)

### Load PT and resave after extracting beh

In [3]:
DATE = 221031
animal = "Pancho"

In [4]:

path = f"/lemur2/lucas/Dropbox/SCIENCE/FREIWALD_LAB/DATA/PopanalTrial/PT-{animal}-{DATE}.pkl"
# path = f"/lemur2/lucas/Dropbox/SCIENCE/FREIWALD_LAB/DATA/PopanalTrial/PT-Pancho-230623.pkl"
import pickle
with open(path, "rb") as f:
    PT = pickle.load(f)


In [ ]:
# [Optional] extract beh then resave
# - beucase this requires data that in on my computer

# Behavioral data for each trial
dfbeh = PT.datasetbeh_extract_df_trialaligned()
dfbeh[:5]

In [ ]:
savedir = f"/lemur2/lucas/Dropbox/SCIENCE/FREIWALD_LAB/DATA/PopanalTrial"
PT.save(savedir, suffix=f"{animal}-{DATE}")

### Testing out file sizes
# # sn = PT.Dat["SN"].values[0]
# sn = PT.Dat["pa"].tolist()

# import pickle
# with open("/tmp/sn.pkl", "wb") as f:
#     pickle.dump(sn, f)

In [ ]:
dfbeh = PT.datasetbeh_extract_df_trialaligned()
dfbeh[dfbeh["aborted"]==True]["trialcode"]


In [ ]:
dfbeh["abort_params"].value_counts()

In [ ]:
idxtrial

In [ ]:
idxtrial = 186
sn, trialsn = PT.extract_sn(idxtrial)


In [ ]:

sn.plotwrapper_raster_oneetrial_multsites(trialsn, list_sites=[1291])

In [ ]:
PT.extract_stroke_peanut_times(idxtrial)

In [ ]:
dfbeh["trial_end_method"].value_counts()

In [ ]:
dfbeh["online_abort"].value_counts()

In [ ]:
dfbeh["motorevents"].value_counts()

In [ ]:
print(row["trial_end_method"])
print(row["online_abort"])

In [ ]:
dfabort = PT.datasetbeh_abort_stroke_times_extract()

In [ ]:
import seaborn as sns
sns.displot(data=dfabort, x="dur_last_stroke_using_peanut", hue="abort_kind_semantic", kind="kde")

In [ ]:
# A stroke failed if the trial is abort and it is the last stroke and eventcode 75 stops during
sn.EventsTimeUsingPhd

### Setting up strokes version of PA

In [14]:
time_pre = 0.4
time_post = 1.2
onset_or_offset = "offset"
PA = PT.pa_extract_align_to_stroke(time_pre, time_post, onset_or_offset)

0
100
200
300
400
500
600
700
800
This many strokes extracted:  2276
DONE!
Appended epoch to self.Dat
Appended character to self.Dat


In [ ]:
# Sanity check, that extracted stroke-level data are correct
dflab = PA.Xlabels["trials"]
# dflab.loc[:, ["trialcode", "stroke_index", "aborted", "trial_end_method", "online_abort"]]
dflab.loc[:, ["trialcode", "stroke_index", "abort_value", "abort_reason"]].to_csv("/tmp/test.csv")
dfbeh.loc[:, ["trialcode", "aborted", "trial_end_method", "online_abort"]].to_csv("/tmp/test2.csv")

In [ ]:
# Plot examples, aligned to strokes

In [15]:
for bregon in ["M1_m", "FP_p", "FP_a"]:
    for chan in PT.info_bregion_to_chans(bregon):
        # PA.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "abort_reason", ["CTXT_shape_prev", "shape", "gridloc"], add_x_zero_line=True);
        fig = PA.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "abort_reason", ["stroke_index", "shape", "gridloc"], add_x_zero_line=True);
        fig.savefig(f"/tmp/{onset_or_offset}-chan-{chan}-{bregon}.pdf")
        plt.close("all")

/home/lucas/miniconda3/envs/drag2_matlab/lib/python3.8/site-packages/numpy/core/_methods.py:269: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/lucas/miniconda3/envs/drag2_matlab/lib/python3.8/site-packages/numpy/core/_methods.py:258: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/home/lucas/miniconda3/envs/drag2_matlab/lib/python3.8/site-packages/numpy/core/_methods.py:269: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/lucas/miniconda3/envs/drag2_matlab/lib/python3.8/site-packages/numpy/core/_methods.py:258: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/home/lucas/miniconda3/envs/drag2_matlab/lib/python3.8/site-packages/numpy/core/_methods.py:269: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/lucas/miniconda3/envs/drag2